In [0]:
%sql
-- 1. Préparation de l'environnement Gold
USE CATALOG dbw_lab;
CREATE SCHEMA IF NOT EXISTS gold;

-- 2. Création de la dimension Clients
CREATE OR REPLACE TABLE gold.dim_customers AS
SELECT 
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM silver.customers;

-- 3. Création de la dimension Vendeurs
CREATE OR REPLACE TABLE gold.dim_sellers AS
SELECT 
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM silver.sellers;

-- 4. Création de la dimension Produits (avec traduction de la catégorie)
CREATE OR REPLACE TABLE gold.dim_products AS
SELECT 
    p.product_id,
    t.product_category_name_english AS category_name,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm
FROM silver.products p
LEFT JOIN silver.product_category_name_translation t 
    ON p.product_category_name = t.product_category_name;

-- 5. Création de la dimension Date
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT DISTINCT
    date(order_purchase_timestamp) AS date_key,
    year(order_purchase_timestamp) AS year,
    month(order_purchase_timestamp) AS month,
    day(order_purchase_timestamp) AS day,
    quarter(order_purchase_timestamp) AS quarter,
    dayofweek(order_purchase_timestamp) AS day_of_week
FROM silver.orders
WHERE order_purchase_timestamp IS NOT NULL;

-- 6. Création de la table de Faits (Ventes / Lignes de commande)
CREATE OR REPLACE TABLE gold.fact_order_items AS
SELECT 
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    o.customer_id,
    date(o.order_purchase_timestamp) AS date_key,
    o.order_status,
    oi.price,
    oi.freight_value,
    (oi.price + oi.freight_value) AS total_item_value
FROM silver.order_items oi
JOIN silver.orders o 
    ON oi.order_id = o.order_id;

In [0]:
%pip install azure-storage-blob python-dotenv

from dotenv import load_dotenv
import os
from azure.storage.blob import BlobServiceClient

# 1. Charger les variables d'environnement
load_dotenv()
storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT")
sas_token = os.getenv("AZURE_SAS_TOKEN")
container_name = "gold" # Cible le conteneur final

# 2. Connexion au service Blob via le jeton SAS
account_url = f"https://{storage_account_name}.blob.core.windows.net?{sas_token}"
blob_service_client = BlobServiceClient(account_url=account_url)
container_client = blob_service_client.get_container_client(container_name)

# 3. Dossier temporaire pour l'exportation
temp_local_dir = "/tmp/gold_exports"
os.makedirs(temp_local_dir, exist_ok=True)

# 4. Liste des tables du modèle en étoile (Schéma Gold)
tables_gold = [
    "dim_customers",
    "dim_sellers",
    "dim_products",
    "dim_date",
    "fact_order_items"
]

for table_name in tables_gold:
    print(f"Préparation et téléversement de la table {table_name}...")
    
    # Lecture depuis la base de données Gold dans Unity Catalog
    df_spark = spark.table(f"dbw_lab.gold.{table_name}")
    
    # Conversion Pandas et sauvegarde locale temporaire
    local_file_path = os.path.join(temp_local_dir, f"{table_name}.csv")
    df_spark.toPandas().to_csv(local_file_path, index=False)
    
    # Téléversement vers Azure dans un dossier structuré
    blob_name = f"star_schema/{table_name}.csv"
    blob_client = container_client.get_blob_client(blob_name)
    
    with open(local_file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)
        
    print(f"✅ {table_name} téléversé avec succès dans le conteneur {container_name} !")

print("🚀 L'intégralité de la couche Gold (Modèle en étoile) a été exportée vers Azure !")